# CSIRO Biomass: EDA + Baseline

Starter notebook for the Kaggle CSIRO Biomass competition. It builds a submission-ready baseline from metadata plus simple image color/texture features.

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

TARGET_WEIGHTS = {
    'Dry_Green_g': 0.1,
    'Dry_Dead_g': 0.1,
    'Dry_Clover_g': 0.1,
    'GDM_g': 0.2,
    'Dry_Total_g': 0.5,
}


def weighted_r2_score(y_true, y_pred, weights):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    weights = np.asarray(weights, dtype=float)
    weighted_mean = np.average(y_true, weights=weights)
    ss_res = np.sum(weights * np.square(y_true - y_pred))
    ss_tot = np.sum(weights * np.square(y_true - weighted_mean))
    return float(1.0 - ss_res / ss_tot) if ss_tot > 0 else 0.0


def find_data_dir():
    candidates = [
        '/kaggle/input/competitions/csiro-biomass',
        '/kaggle/input/csiro-biomass',
        os.environ.get('CSIRO_DATA_DIR'),
        Path.cwd() / 'data' / 'csiro-biomass',
        Path.cwd().parent / 'data' / 'csiro-biomass',
        'data/csiro-biomass',
    ]
    for candidate in candidates:
        if candidate is None:
            continue
        path = Path(candidate)
        if (path / 'train.csv').exists() and (path / 'test.csv').exists():
            return path
    raise FileNotFoundError('Expected competition data at /kaggle/input/competitions/csiro-biomass')


def add_calendar_features(df, date_col='Sampling_Date'):
    out = df.copy()
    if date_col not in out.columns:
        return out
    dates = pd.to_datetime(out[date_col], errors='coerce')
    out['sample_month'] = dates.dt.month
    out['sample_dayofyear'] = dates.dt.dayofyear
    out['sample_year'] = dates.dt.year
    return out


def extract_image_features(image_paths, image_size=256):
    from PIL import Image, ImageStat
    from tqdm.auto import tqdm

    rows = []
    for path in tqdm(image_paths, desc='image features'):
        row = {'image_path': str(path)}
        try:
            image = Image.open(path).convert('RGB')
            row['width'], row['height'] = image.size
            small = image.resize((image_size, image_size))
            arr = np.asarray(small, dtype=np.float32) / 255.0
            stat = ImageStat.Stat(small)
            for idx, channel in enumerate('rgb'):
                row[f'{channel}_mean'] = float(stat.mean[idx] / 255.0)
                row[f'{channel}_std'] = float(stat.stddev[idx] / 255.0)
                row[f'{channel}_p10'] = float(np.quantile(arr[:, :, idx], 0.10))
                row[f'{channel}_p50'] = float(np.quantile(arr[:, :, idx], 0.50))
                row[f'{channel}_p90'] = float(np.quantile(arr[:, :, idx], 0.90))
            green = arr[:, :, 1]
            red = arr[:, :, 0]
            blue = arr[:, :, 2]
            row['excess_green'] = float(np.mean(2 * green - red - blue))
            row['green_red_ratio'] = float(np.mean(green / (red + 1e-4)))
            row['visible_ndvi_proxy'] = float(np.mean((green - red) / (green + red + 1e-4)))
            row['brightness'] = float(np.mean(arr))
            row['contrast'] = float(np.std(arr))
        except Exception as exc:
            row['image_error'] = str(exc)
        rows.append(row)
    return pd.DataFrame(rows)

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')

## Load data

In [ ]:
DATA_DIR = find_data_dir()
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
print(f'Data: {DATA_DIR}')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(train.shape, test.shape, sample_submission.shape)
display(train.head())
display(test.head())

In [ ]:
target_order = list(TARGET_WEIGHTS)
display(train['target_name'].value_counts().reindex(target_order))
display(train.groupby('target_name')['target'].describe().reindex(target_order))

## Image and tabular features

In [ ]:
def image_feature_table(df, cache_name):
    cache_path = OUTPUT_DIR / cache_name
    if cache_path.exists():
        return pd.read_csv(cache_path)
    unique_paths = df['image_path'].drop_duplicates().tolist()
    full_paths = [DATA_DIR / p for p in unique_paths]
    feats = extract_image_features(full_paths)
    feats['image_path'] = unique_paths
    feats.to_csv(cache_path, index=False)
    return feats

train_img = image_feature_table(train, 'train_image_features.csv')
test_img = image_feature_table(test, 'test_image_features.csv')

display(train_img.head())

In [ ]:
train_base = train.merge(train_img, on='image_path', how='left')
test_base = test.merge(test_img, on='image_path', how='left')

train_base = add_calendar_features(train_base)
test_base = add_calendar_features(test_base)

feature_cols = [
    'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm',
    'sample_month', 'sample_dayofyear', 'sample_year',
    'width', 'height', 'r_mean', 'r_std', 'r_p10', 'r_p50', 'r_p90',
    'g_mean', 'g_std', 'g_p10', 'g_p50', 'g_p90',
    'b_mean', 'b_std', 'b_p10', 'b_p50', 'b_p90',
    'excess_green', 'green_red_ratio', 'visible_ndvi_proxy', 'brightness', 'contrast',
]
feature_cols = [c for c in feature_cols if c in train_base.columns]
categorical_cols = [c for c in ['State', 'Species'] if c in feature_cols]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

preprocess = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=3)),
        ]), categorical_cols),
    ],
    remainder='drop',
)

feature_cols

## Cross-validation

In [ ]:
def make_model(random_state=42):
    return Pipeline([
        ('preprocess', preprocess),
        ('model', HistGradientBoostingRegressor(
            loss='squared_error',
            learning_rate=0.06,
            max_iter=350,
            l2_regularization=0.05,
            random_state=random_state,
        )),
    ])


oof_parts = []
groups = train_base['image_path']
gkf = GroupKFold(n_splits=5)

for fold, (trn_idx, val_idx) in enumerate(gkf.split(train_base, groups=groups), start=1):
    fold_preds = []
    for target_name in target_order:
        trn = train_base.iloc[trn_idx].query('target_name == @target_name')
        val = train_base.iloc[val_idx].query('target_name == @target_name')
        model = make_model(random_state=1000 + fold)
        model.fit(trn[feature_cols], trn['target'])
        pred = np.clip(model.predict(val[feature_cols]), 0, None)
        fold_preds.append(pd.DataFrame({
            'sample_id': val['sample_id'].values,
            'image_path': val['image_path'].values,
            'target_name': target_name,
            'target': val['target'].values,
            'prediction': pred,
        }))
    fold_df = pd.concat(fold_preds, ignore_index=True)
    fold_df['weight'] = fold_df['target_name'].map(TARGET_WEIGHTS)
    score = weighted_r2_score(fold_df['target'], fold_df['prediction'], fold_df['weight'])
    mae = mean_absolute_error(fold_df['target'], fold_df['prediction'])
    print(f'fold {fold}: weighted R2={score:.5f} MAE={mae:.3f}')
    oof_parts.append(fold_df)

oof = pd.concat(oof_parts, ignore_index=True)
overall_score = weighted_r2_score(oof['target'], oof['prediction'], oof['weight'])
print(f'OOF weighted R2: {overall_score:.5f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
sns.scatterplot(data=oof, x='target', y='prediction', hue='target_name', s=18, ax=ax)
limit = max(oof['target'].max(), oof['prediction'].max())
ax.plot([0, limit], [0, limit], color='black', linewidth=1)
ax.set_title(f'OOF predictions, weighted R2={overall_score:.4f}')
plt.show()

## Train full models and create submission

In [ ]:
test_predictions = []
models = {}

for target_name in target_order:
    trn = train_base.query('target_name == @target_name')
    tst = test_base.query('target_name == @target_name')
    model = make_model(random_state=2026)
    model.fit(trn[feature_cols], trn['target'])
    pred = np.clip(model.predict(tst[feature_cols]), 0, None)
    models[target_name] = model
    test_predictions.append(pd.DataFrame({
        'sample_id': tst['sample_id'].values,
        'target_name': target_name,
        'target': pred,
    }))

submission = pd.concat(test_predictions, ignore_index=True)
submission = sample_submission[['sample_id']].merge(submission[['sample_id', 'target']], on='sample_id', how='left')

fallback = train.groupby('target_name')['target'].median()
test_targets = test[['sample_id', 'target_name']].drop_duplicates()
missing = submission['target'].isna()
if missing.any():
    missing_targets = submission.loc[missing, ['sample_id']].merge(test_targets, on='sample_id', how='left')['target_name']
    submission.loc[missing, 'target'] = missing_targets.map(fallback).values

submission['target'] = submission['target'].clip(lower=0)
submission_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)
display(submission.head(10))
print(submission.shape)
print(f'Wrote {submission_path}')